# Week 3 SQL & PostgreSQL — Member 1 Teaching Notebook

**Your topic:** `SELECT`, `WHERE`, and `NULL` behaviour  
**Your question-round day:** Monday  
**Your Friday mini-project task:** Query books with no copies on the shelf.

## Learning goals
By the end, you should be able to:
1. Explain what a table, row, and column are.
2. Use `SELECT` to retrieve all columns or selected columns.
3. Use `WHERE` to filter rows.
4. Distinguish a recorded zero from a missing/unknown value (`NULL`).
5. Use `IS NULL` and `IS NOT NULL` correctly.
6. Explain why `= NULL` is not the right test.

> **Important project note:** The README gives the Friday task but does not specify the final table or column names. This notebook uses an illustrative `books` table and `copies_on_shelf` column. Check your team's actual schema before using the query in the project.


## 1. A simple picture of a database table

A **table** stores related information. A **row** is one record. A **column** is one kind of information about each record.

We will practise with fictional book data.


In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory practice database.
# SQLite is used here so the notebook runs without a PostgreSQL server.
# The SELECT / WHERE / IS NULL examples below are also valid PostgreSQL syntax.
connection = sqlite3.connect(":memory:")

connection.execute("""
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author TEXT,
    copies_on_shelf INTEGER
);
""")

sample_books = [
    (1, "Things Fall Apart", "Chinua Achebe", 5),
    (2, "Purple Hibiscus", "Chimamanda Adichie", 0),
    (3, "Half of a Yellow Sun", "Chimamanda Adichie", 3),
    (4, "The River Between", "Ngũgĩ wa Thiong'o", None),
    (5, "Weep Not, Child", "Ngũgĩ wa Thiong'o", 2),
]

connection.executemany(
    "INSERT INTO books (id, title, author, copies_on_shelf) VALUES (?, ?, ?, ?)",
    sample_books
)
connection.commit()

pd.read_sql_query("SELECT * FROM books;", connection)


## 2. SELECT — choose what information to retrieve

`SELECT` tells SQL which **columns** you want to see.

### Select every column
```sql
SELECT *
FROM books;
```
The asterisk (`*`) means all columns.

### Select only chosen columns
```sql
SELECT title, author
FROM books;
```
This returns only the title and author columns. It does not delete or change the other columns in the table.


In [ ]:
pd.read_sql_query(
    "SELECT title, author FROM books;",
    connection
)

: 

### Checkpoint 1

In your own words: what does `SELECT title, author FROM books;` ask the database to show?

**Model answer:** Show the `title` and `author` columns for the rows in the `books` table.


## 3. WHERE — filter which rows you want

`WHERE` adds a condition. SQL returns only rows where the condition is **TRUE**.

Example: find books by Chinua Achebe.
```sql
SELECT *
FROM books
WHERE author = 'Chinua Achebe';
```


In [ ]:
pd.read_sql_query(
    """
    SELECT *
    FROM books
    WHERE author = 'Chinua Achebe';
    """,
    connection
)

: 

### More WHERE examples

Find books with exactly zero copies:
```sql
SELECT title, copies_on_shelf
FROM books
WHERE copies_on_shelf = 0;
```

Find books with more than two copies:
```sql
SELECT title, copies_on_shelf
FROM books
WHERE copies_on_shelf > 2;
```

Text values normally go inside single quotation marks. Numeric values such as `0` do not need quotation marks.


In [ ]:
pd.read_sql_query(
    """
    SELECT title, copies_on_shelf
    FROM books
    WHERE copies_on_shelf = 0;
    """,
    connection
)

## 4. NULL — a missing or unknown value

`NULL` means a value is missing, unknown, or not recorded. It is **not the same as zero** and is not the same as an empty string.

In our sample data:
- `Purple Hibiscus` has `0` copies: the count is known and is zero.
- `The River Between` has `NULL`: the count is not known/recorded.

### Correct way to find missing values
```sql
SELECT *
FROM books
WHERE copies_on_shelf IS NULL;
```


In [ ]:
pd.read_sql_query(
    """
    SELECT title, copies_on_shelf
    FROM books
    WHERE copies_on_shelf IS NULL;
    """,
    connection
)

### Correct way to find values that are present

```sql
SELECT *
FROM books
WHERE copies_on_shelf IS NOT NULL;
```


In [ ]:
pd.read_sql_query(
    """
    SELECT title, copies_on_shelf
    FROM books
    WHERE copies_on_shelf IS NOT NULL;
    """,
    connection
)

## 5. Why `= NULL` is wrong

Do **not** write:
```sql
WHERE copies_on_shelf = NULL
```

Use:
```sql
WHERE copies_on_shelf IS NULL
```

SQL comparisons involving `NULL` produce **UNKNOWN**, not TRUE or FALSE. SQL uses three-valued logic:
- `TRUE`
- `FALSE`
- `UNKNOWN`

A `WHERE` clause keeps only rows for which the condition is TRUE. Rows where the condition is FALSE or UNKNOWN are not returned.

This is why `= NULL` does not find the missing values.


## 6. Connect your topic to Friday's mini-project

The README says your task is: **“Query books with no copies on the shelf.”**

There is an important question to ask your team: what does “no copies” mean in the agreed data?

### If it means a known count of zero
```sql
SELECT *
FROM books
WHERE copies_on_shelf = 0;
```

### If it means the count has not been recorded
```sql
SELECT *
FROM books
WHERE copies_on_shelf IS NULL;
```

Do not silently treat missing data as zero. Confirm the team's schema and meaning first. The actual table/column names may differ from this practice example.


## 7. Practice exercises

Try each exercise before opening the answer cell below it.


### 1. Show every column and every row from `books`.
<details><summary>Show a model answer</summary>

```sql
SELECT * FROM books;
```

</details>

### 2. Show only `title` and `author`.
<details><summary>Show a model answer</summary>

```sql
SELECT title, author FROM books;
```

</details>

### 3. Find books written by Chimamanda Adichie.
<details><summary>Show a model answer</summary>

```sql
SELECT * FROM books WHERE author = 'Chimamanda Adichie';
```

</details>

### 4. Find books with exactly zero recorded copies.
<details><summary>Show a model answer</summary>

```sql
SELECT * FROM books WHERE copies_on_shelf = 0;
```

</details>

### 5. Find books whose copy count is missing.
<details><summary>Show a model answer</summary>

```sql
SELECT * FROM books WHERE copies_on_shelf IS NULL;
```

</details>

### 6. Find books whose copy count is present.
<details><summary>Show a model answer</summary>

```sql
SELECT * FROM books WHERE copies_on_shelf IS NOT NULL;
```

</details>

## 8. A short teaching script for your Monday question round

You can explain the topic in this order:

1. “A table stores records in rows and information in columns.”
2. “`SELECT` chooses the columns we want to retrieve.”
3. “`WHERE` filters the rows according to a condition.”
4. “`NULL` represents a missing or unknown value; it is not zero.”
5. “We test missing values with `IS NULL`, and present values with `IS NOT NULL`.”
6. “We do not use `= NULL`, because comparisons with NULL evaluate to UNKNOWN.”
7. “For the Friday task, we must confirm whether ‘no copies’ means a known zero or a missing count.”

### Trade-off / limitation to mention
A query using `SELECT *` is convenient while exploring a table, but selecting only the needed columns is clearer and can reduce unnecessary data transfer. Also, `NULL` correctly preserves uncertainty, but the application must decide how missing counts should be handled rather than assuming they are zero.


## 9. Quick self-test

Answer without looking back:
1. What does `*` mean in `SELECT *`?
2. What does `WHERE` do?
3. Is `NULL` the same as zero?
4. Which operator checks for missing values?
5. Why is `= NULL` incorrect?
6. What is the difference between `= 0` and `IS NULL`?

## 10. PostgreSQL note

This notebook runs the practice queries with SQLite so you can start immediately without configuring a server. The core syntax used for `SELECT`, `WHERE`, `IS NULL`, and `IS NOT NULL` is also PostgreSQL-compatible. Your team's actual project should be tested against PostgreSQL and its agreed schema.
